# Performance

<!-- Drafted with the help of an AI assistant (Claude, Anthropic), disclosed per course rules. -->

Running time, memory and output size of the parser, from `data/m2/m2_benchmark_summary.csv` and `data/m2/m2_resource_timeseries.csv`, produced by `python -m src.benchmark.m2_benchmark`.

:::{warning}
**These are not the official benchmark.** The files hold one run of the real pipeline on the 100-paper M1 smoke crawl, on a **local Windows machine** (16 logical CPUs, 33.6 GB RAM) with 2 worker processes. The statement asks for a final run on a Google Colab CPU-only instance; its CSV files replace these without changing this page. Nothing here may be read as the performance on Colab or on the full corpus.
:::

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA = Path("../data/m2")
summary = pd.read_csv(DATA / "m2_benchmark_summary.csv")
series = pd.read_csv(DATA / "m2_resource_timeseries.csv")
papers = pd.read_csv(DATA / "m2_papers.csv", dtype={"arxiv_id": str})

## Summary

*End-to-end wall time* is measured around the whole run (start of the process to its exit). *Time per publication* is the wall time divided by the publications of the corpus. *Maximum RAM* is the largest sampled resident memory of the runner and all its worker processes together; the sampler looks every half second, so a shorter spike can be missed. *Output size* is the size of the Parquet shards.

In [ ]:
summary.set_index('metric')

## RAM and output size during the run

In [ ]:
long = series.melt(id_vars="t_sec", var_name="measure", value_name="MB")
fig = px.line(
    long,
    x="t_sec",
    y="MB",
    color="measure",
    labels={"t_sec": "Seconds since the start", "MB": "MB"},
    title="Resident memory of the process tree and size of the output folder",
)
fig

## Where the time goes

`elapsed_s` is the time one worker spent on one publication (all its versions, from reading the files to the rows of the three tables).

In [ ]:
fig = px.histogram(
    papers,
    x="elapsed_s",
    nbins=40,
    log_y=True,
    labels={"elapsed_s": "Seconds per publication"},
    title="Time spent on one publication",
)
fig.update_layout(yaxis_title="Publications (log scale)")
fig

In [ ]:
papers["elapsed_s"].describe(percentiles=[0.5, 0.9, 0.99]).round(3).to_frame("seconds")

## What this does not tell

A mostly single-version smoke sample is cheaper per publication than a corpus in which every second paper has two or more versions, so a linear extrapolation to 20,000 publications from these numbers would be indicative at best; the official run on Colab is the measurement. Memory grows with the size of a shard (100 publications by default, `--shard-papers`), not with the corpus, and `--workers` is capped at 2 in the plan so that a 13.6 GB Colab runtime stays well inside its limit.